# Step 2: attach features to trade entries, point in time

Each trade entry gets the features from the last *completed* bar of each timeframe at the entry time. `merge_asof` with `direction='backward'` and a one-bar shift makes sure no entry sees a bar that had not closed yet.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR (relative to the repo by default).
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import numpy as np
import pandas as pd

features = pd.read_csv(OUT_DIR / "features.csv", parse_dates=["date"])
BAR = {"H1": pd.Timedelta("1h"), "H4": pd.Timedelta("4h"), "D1": pd.Timedelta("1D")}
features["available_at"] = features.date + features.timeframe.map(BAR)  # a bar is usable once it closes

rng = np.random.default_rng(SEED)
h1 = features[features.timeframe == "H1"]
entries = h1.sample(800, random_state=SEED)[["symbol", "date"]].rename(columns={"date": "entry_time"})
entries["entry_time"] += pd.to_timedelta(rng.integers(0, 60, len(entries)), unit="min")
entries["direction"] = rng.choice([1, -1], len(entries))
# Synthetic outcome loosely tied to trend so the model in step 3 has something to find.
entries = entries.sort_values("entry_time").reset_index(drop=True)

In [ ]:
FEATURE_COLS = ["stoch_k", "stoch_up", "ema_trend_up", "dist_ema_200", "rsi_14", "atr_14_pct", "bb_position", "ret_1", "ret_5", "ret_20"]
merged = entries.copy()
for tf in BAR:
    f = (features[features.timeframe == tf][["symbol", "available_at"] + FEATURE_COLS]
         .rename(columns={c: f"{tf}_{c}" for c in FEATURE_COLS}).sort_values("available_at"))
    merged = pd.merge_asof(merged.sort_values("entry_time"), f, left_on="entry_time", right_on="available_at",
                           by="symbol", direction="backward").drop(columns="available_at")

logit = 1.5 * merged.direction * merged["D1_ema_trend_up"].sub(0.5) + rng.normal(0, 1, len(merged))
merged["win"] = (logit > 0).astype(int)
merged = merged.dropna()
merged.to_csv(OUT_DIR / "model_dataset.csv", index=False)
merged.shape